# DiscoveryStack SEO/GEO v4-500：AI 接手 Notebook

> **目前狀態：fast-path evaluated / candidate_not_ready。** 固定 500 筆資料已通過驗證；本次在 Tesla T4 以 `stage_branch_weighted`、seed `20260820`、最多 2 epochs 完成 bounded fast-path、final evaluation 與 owner-only artifact upload。但這不是完整 2 configs × 3 seeds ablation，且 test_v2 有三個 journeyStage 類別 predicted support 為零，因此不可視為 production-ready。

本 Notebook 是交接說明，不會自動載入、顯示或上傳原始 JSONL。1,087 筆擴充方向已放棄，後續只允許使用固定的 manifest-v4-500。


## 1. 已固定的資料版本

資料由兩個來源家族組成：250 筆 Google Search Central legacy，以及 250 筆 web.dev 公開文件清理後衍生樣本。固定摘要為：500 rows；manifest hash `d1868ebd13ebf5b489e551afba2c047c19af5022e83c101e23a9927beaf02977`；dataset digest `6aaf9e6c57f4d930ba220575bc1a5f7cb0ba6145373e7a8b629f89403c85c474`；splits 為 train 350、validation 75、test_legacy_v1 32、test_v2 43。

journeyStage totals 為 conversion 83、discovery 89、progression 117、response 118、understanding 93。review state 為 reviewed 250、needs_adjudication 250；因此此版本是 development candidate，不是 production-ready。


In [ ]:
# Safe metadata-only check. The raw manifest is intentionally not part of this repository.
from pathlib import Path
SAFE_PROJECT_ROOT = Path('/home/ubuntu/DiscoveryStack_nuxt')
PRIVATE_MANIFEST = Path('/home/ubuntu/private_training/discoverystack-manifest-v4-500.jsonl')
print({
    'safe_project_root_exists': SAFE_PROJECT_ROOT.exists(),
    'private_manifest_expected_at': str(PRIVATE_MANIFEST),
    'raw_manifest_must_not_be_committed': True,
})


## 2. 清洗與治理

收集流程是：以公開官方來源及 robots／sitemap 為入口，逐頁保存來源證據；抽取正文與核准的結構化欄位；移除 HTML boilerplate、script/style、導航重複、明顯重複段落與 PII；建立 deterministic `trainingText`；計算 hash、canonical domain hash 與 near-duplicate cluster；依 journey taxonomy 建立 targets 與 `stageEvidence`；最後通過 rights、robots、PII、dedupe、review、split isolation 及 digest 的 fail-closed 驗證。

資料契約在 `ml/schemas/dataset-v4-500.schema.json`。`stageEvidence` 是治理與 provenance 證據，不是 model feature。新增的 web.dev 250 筆仍有 `needs_adjudication`，生產前需要人工 adjudication。


## 3. Y：九項任務與標籤

`journeyStage` 與 `actionPriority` 是 single-label；`searchIntents`、`contentTypes`、`audienceRoles`、`geoSignals`、`citationReadiness`、`technicalSeoSignals`、`frictionSignals` 是 multi-label。列中的 `targets` 是 supervised learning 的 Y。`secondaryStages`、`stageCueTypes`、`stageEvidence`、post-outcome data 與 test metrics 都禁止作為 journeyStage feature。完整 label vocabulary 位於本機私有的安全統計檔 `handoff_inventory_500.json`，不含樣本文字。


## 4. X：文字與 inference-safe stage branch

文字 X 是 `trainingText`，使用 `distilbert-base-multilingual-cased`、MAX_LENGTH 256、truncation 與 max padding。journeyStage 另使用從同一份 inference-time `trainingText` 重建的 14 維 cue subset：problem statement、question、definition、how-it-works、comparison、requirements、troubleshooting、error/debug、remediation、CTA、contact/purchase、form presence、log text length、log heading/newline count。

`ml/schemas/features-v1.json` 宣告了更寬的 contract，還包含 price/shipping/returns、approved structured extract、page type one-hot、schema type one-hot 與 evidence quality vector；但目前 optimized Notebook 實際只使用上述 14 維。所有 mean/std 只在 train fit，再 freeze 到 validation、test 與 inference。


## 5. 模型、訓練與選模

共享 DistilBERT first-token pooled representation。journeyStage head 將 pooled text 與 64 維 feature MLP embedding concat；其他八個 heads 只用 pooled text。AdamW 使用 learning rate `2e-5`、weight decay `0.01`、batch 8、最多 8 epochs、gradient clipping 1.0、early stopping patience 2。journeyStage loss multiplier 為 2.0，class weights 使用 train-only sqrt inverse-frequency。既定候選為 `text_only_baseline` 與 `stage_branch_weighted`，三個 seeds `20260820/20260821/20260822`；只以 validation journeyStage macro-F1 選模，不能用 test_v2 調參。


## 6. 評估與 artifact gates

只有在看到 epoch logs、`run_records`、checkpoint 與 selected path 後，才可評估 test_v2 與固定的 test_legacy_v1。必須輸出 journeyStage macro/micro F1、per-class F1、confusion matrix、predicted support 與 zero-prediction classes。test_v2 任一 journeyStage 類別 predicted support 為零時，狀態是 `candidate_not_ready`；即使沒有零支持，也最多是 `candidate_ready_for_review`，不等同 production-ready。

allow-list artifact 只能含 self-describing checkpoint／safetensors、tokenizer、training config、label maps、feature stats、metrics、兩個 test predictions、manifest 與 checksums。禁止 raw JSONL、HTML、browser profile、OAuth material、`.env`、token 或 secrets。


## 6A. 本次 fast-path 實際結果

本次只執行 `stage_branch_weighted`、seed `20260820`、最多 2 epochs；epoch 1 trainLoss `7.8520873135`、validation macro-F1 `0.0898395722`，epoch 2 trainLoss `6.3885995041`、validation macro-F1 `0.2274165977`，並選取 epoch 2。

| split | macro-F1 | micro-F1 | predicted support | readiness |
|---|---:|---:|---|---|
| `test_v2` (43) | 0.1879598662 | 0.3953488372 | conversion 0 / discovery 11 / progression 0 / response 32 / understanding 0 | `candidate_not_ready` |
| `test_legacy_v1` (32) | 0.0422222222 | 0.0625000000 | conversion 13 / discovery 0 / progression 2 / response 0 / understanding 17 | regression fail signal |

由於 test_v2 有 conversion、progression、understanding 三個零預測類別，artifact 必須保持 `candidate_not_ready`。完整 evaluation JSON、predictions、checkpoint 與 artifact manifest 僅在 owner-only artifact 內，未嵌入本 Notebook。


## 7. 本次阻塞與接手方法

Colab 曾取得 Tesla T4，Drive OAuth 讀取成功，500-row validation 成功；阻塞是 runtime 的 cell execution state 與 Notebook source 不同步。最後具體錯誤是 `NameError: name 'Path' is not defined`，發生在 `run_root = Path('/content/optimized_runs_v4')`。GUI 插入的 `path_ready` 修復 cell 沒有可驗證 output，因此不能視為成功。後來已在本機產生清除輸出的 recovery Notebook，但使用者已要求停止訓練。

本次 recovery retry 已經依序通過 `imports_ready`、500-row digest／split validation、`model_definition_ready`、`smoke_train_batch_ready` 與 `lossFinite=True`，再執行 bounded fast path。日後若要取得正式比較結果，仍須在新鮮 T4 runtime 重新執行完整 2 configs × 3 seeds；本次 fast-path 的 checkpoint、metrics 與 artifact 只能作為 baseline。


## 8. 檔案位置與私有邊界

專案內可提交的安全文件包括 `ml/AI_HANDOFF_500_TRAINING.md`、本 Notebook、`ml/schemas/`、`ml/runbooks/` 與經清除輸出的 recovery Notebook。私有原始 manifest 位於 `/home/ubuntu/private_training/discoverystack-manifest-v4-500.jsonl`，owner-only Drive snapshot ID 為 `1i2N36qNgHOmx1PM7-DZV-R02wq73t3SV`；recovery Notebook ID 為 `1ZIgD4P0Zs-MOqdoWkAQJ72FI92E60DUS`。本次 fast-path artifact ZIP `discoverystack-ml-v4-500-d1868ebd13eb.zip` 為 500810915 bytes，SHA256 為 `89683d0630aff1e003cad360e44ae255f3434bbbb0e07af01d1057990e8443ad`，owner-only Drive file ID 為 `1yBQj_VD8g0jGo1c7UbpTnpb1fzTucMI3`；其 readiness 是 `candidate_not_ready`。


In [ ]:
# Handoff smoke check: this verifies only safe project files and never reads raw row text.
from pathlib import Path
root = Path('/home/ubuntu/DiscoveryStack_nuxt')
required = [
    root / 'ml/AI_HANDOFF_500_TRAINING.md',
    root / 'ml/DiscoveryStack_500_AI_HANDOFF.ipynb',
    root / 'ml/schemas/dataset-v4-500.schema.json',
    root / 'ml/schemas/features-v1.json',
    root / 'ml/runbooks/500_EXECUTION_RUNBOOK.md',
]
print({'required_safe_files_present': all(p.exists() for p in required), 'count': len(required)})


## 9. 結論

本次已完成固定 v4-500 資料的可追溯 fast-path training、evaluation 與私有 artifact 保存，但不是完整 ablation，也不是 production-ready 模型。test_v2 journeyStage macro-F1 為 `0.1879598662`、micro-F1 為 `0.3953488372`，conversion、progression、understanding 的 predicted support 為零；test_legacy_v1 macro-F1 為 `0.0422222222`。任何正式完成聲明仍必須補上完整兩 config、三 seed、checkpoint reload、legacy regression 與 zero-prediction gate 證據。
